<a href="https://colab.research.google.com/github/Wajahat725/EchoLock/blob/main/echolock.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Introduction


# EchoLock: Improved Session Password-Based Security System

**Objective:** Trains a model that estimates whether an attacker who has watched a user's past
login sessions can identify the secret character at a given click position.

---
### Methodology
1. **Data Loading:** Load session logs and user secrets.
2. **Preprocessing:** Filter successful sessions and aggregate observations for the first 5 sessions.
3. **Feature Engineering:** Calculate frequency and count of each observed symbol per click rank.
4. **Model Training:** Train a Logistic Regression classifier to predict if the observed symbol is the actual secret.
5. **Evaluation:** Assess model accuracy against a majority-vote baseline.

 Import Libraries & Setup

In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression

N_SESSIONS = 5
SYMBOL_POOL = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789")

Data Loading

In [2]:
from google.colab import files

uploaded = files.upload()

sessions = pd.read_csv("login_sessions.csv")
secrets = pd.read_csv("secrets.csv")

print(f"Sessions: {sessions.shape[0]} rows")
print(f"Secrets: {secrets.shape[0]} users")
sessions.head()

Saving secrets (1).csv to secrets (1).csv
Saving login_sessions (2).csv to login_sessions (2).csv
Sessions: 7200 rows
Secrets: 60 users


,user_id,session_id,timestamp,click_rank,secret_length,clicked_position,grid_layout_36,true_symbol_at_position,attacker_observed_symbol,session_success
0,U001,S000001,1700137236,1,6,28,1BIJPMQOU0K47Z6A5SF2E8NRYLD3HVGW9XTC,H,H,True
1,U001,S000001,1700137236,2,6,1,1BIJPMQOU0K47Z6A5SF2E8NRYLD3HVGW9XTC,B,B,True
2,U001,S000001,1700137236,3,6,23,1BIJPMQOU0K47Z6A5SF2E8NRYLD3HVGW9XTC,R,O,True
3,U001,S000001,1700137236,4,6,4,1BIJPMQOU0K47Z6A5SF2E8NRYLD3HVGW9XTC,P,Z,True
4,U001,S000001,1700137236,5,6,7,1BIJPMQOU0K47Z6A5SF2E8NRYLD3HVGW9XTC,O,R,True


Data Preprocessing & Feature Engineering

In [3]:
sessions = sessions[sessions.session_success == True].sort_values(
    ["user_id", "timestamp", "click_rank"]
)
secret_lookup = dict(zip(secrets.user_id, secrets.secret_sequence))

rows = []
for (user_id, rank), group in sessions.groupby(["user_id", "click_rank"]):
    obs = group["attacker_observed_symbol"].astype(str).tolist()[:N_SESSIONS]
    obs = [o for o in obs if o not in ("", "nan")]
    total_obs = len(obs)
    if total_obs == 0:
        continue

    counts = {s: obs.count(s) for s in set(obs)}
    secret_seq = secret_lookup.get(user_id)
    if not secret_seq:
        continue

    true_symbol = secret_seq[rank - 1] if rank - 1 < len(secret_seq) else None

    for candidate in SYMBOL_POOL:
        count = counts.get(candidate, 0)
        rows.append({
            "user_id": user_id,
            "click_rank": rank,
            "secret_length": len(secret_seq),
            "candidate": candidate,
            "count": count,
            "total_obs": total_obs,
            "freq": count / total_obs,
            "label": int(candidate == true_symbol),
        })

features_df = pd.DataFrame(rows)
print(f"Feature table: {features_df.shape[0]} rows")
features_df.head()

Feature table: 10368 rows


,user_id,click_rank,secret_length,candidate,count,total_obs,freq,label
0,U001,1,6,A,0,5,0.0,0
1,U001,1,6,B,0,5,0.0,0
2,U001,1,6,C,0,5,0.0,0
3,U001,1,6,D,0,5,0.0,0
4,U001,1,6,E,0,5,0.0,0


Train and Test Split

In [4]:
FEATURES = ["click_rank", "secret_length", "count", "total_obs", "freq"]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(features_df, groups=features_df["user_id"]))

train_df = features_df.iloc[train_idx]
test_df = features_df.iloc[test_idx]

X_train, y_train = train_df[FEATURES], train_df["label"]
X_test, y_test = test_df[FEATURES], test_df["label"]

print(f"Train users: {train_df.user_id.nunique()}, test users: {test_df.user_id.nunique()}")
print(f"Train rows: {len(X_train)}, test rows: {len(X_test)}")

Train users: 45, test users: 15
Train rows: 7848, test rows: 2520


Model Training

In [5]:
model = LogisticRegression(class_weight="balanced", max_iter=1000)
model.fit(X_train, y_train)
print("Model trained.")

Model trained.


 Evaluation & Results

In [6]:
test_df = test_df.copy()
test_df["pred_proba"] = model.predict_proba(X_test)[:, 1]

correct_model = 0
correct_baseline = 0
total_groups = 0

for (user_id, rank), group in test_df.groupby(["user_id", "click_rank"]):
    total_groups += 1
    model_pick = group.loc[group["pred_proba"].idxmax(), "candidate"]
    baseline_pick = group.loc[group["count"].idxmax(), "candidate"]
    true_pick = group.loc[group["label"] == 1, "candidate"]
    true_pick = true_pick.iloc[0] if len(true_pick) else None

    correct_model += int(model_pick == true_pick)
    correct_baseline += int(baseline_pick == true_pick)

print(f"N_SESSIONS: {N_SESSIONS}")
print(f"Majority-vote baseline accuracy: {correct_baseline / total_groups:.3f}")
print(f"Logistic regression accuracy:    {correct_model / total_groups:.3f}")
print(f"Evaluated on {total_groups} user/rank groups")

N_SESSIONS: 5
Majority-vote baseline accuracy: 0.957
Logistic regression accuracy:    0.957
Evaluated on 70 user/rank groups


In [7]:
joblib.dump(model, "leakage_model.pkl")
print("Saved leakage_model.pkl")

from google.colab import files
files.download("leakage_model.pkl")

Saved leakage_model.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>